# LeetCode 1523 — Count Odd Numbers in an Interval Range bằng Python

**Link đề:** https://leetcode.com/problems/count-odd-numbers-in-an-interval-range/
**Độ khó:** Easy · **Chủ đề:** Math

```python
class Solution:
    def countOdds(self, low: int, high: int) -> int:
        ...
```

### Mục tiêu học
1. Hiểu đoạn **bao gồm hai đầu** `[low, high]` và cách đếm số phần tử `high - low + 1`.
2. Kiểm tra chẵn/lẻ bằng `% 2` và `& 1`.
3. Thấy vì sao vòng lặp `O(n)` **không chạy nổi** với `high` tới `10^9`.
4. Rút ra công thức `O(1)` bằng 3 cách nhìn: "đếm số lẻ từ 0" (`(x + 1) // 2`), phân tích theo chẵn/lẻ hai đầu (gợi ý của đề), và `len(range(...))`.
5. Trace, chứng minh, test đối chiếu và áp dụng cho biến thể (đếm bội của `k`, tổng các số lẻ…).

**Cách học:** đọc markdown → đoán output của cell code kế tiếp → chạy → so sánh. Các cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài

Cho hai số nguyên **không âm** `low` và `high`. Trả về số lượng số **lẻ** nằm giữa `low` và `high` (**tính cả hai đầu**).

| low | high | Các số lẻ | Output |
|---:|---:|---|---:|
| 3 | 7 | 3, 5, 7 | 3 |
| 8 | 10 | 9 | 1 |

**Constraints:** `0 <= low <= high <= 10^9`.

Câu hỏi đặt ra:
- Đoạn `[3, 7]` có bao nhiêu số? (`7 - 3 + 1 = 5`, không phải 4.)
- Duyệt từng số tới `10^9` có kịp không?
- Có công thức trực tiếp không?

In [1]:
low, high = 3, 7
print("các số trong đoạn:", list(range(low, high + 1)))
print("số phần tử:", high - low + 1)
print("các số lẻ:", [x for x in range(low, high + 1) if x % 2 == 1])

các số trong đoạn: [3, 4, 5, 6, 7]
số phần tử: 5
các số lẻ: [3, 5, 7]


## 2. Nền tảng Python cần biết

### 2.1. `range(a, b)` không lấy `b` → muốn lấy cả `high` phải viết `range(low, high + 1)`

In [2]:
print(list(range(8, 10)))       # thiếu 10
print(list(range(8, 10 + 1)))   # đủ 8, 9, 10

[8, 9]
[8, 9, 10]


### 2.2. Chẵn / lẻ: `x % 2` và `x & 1`

- `x % 2` là số dư khi chia 2: `0` nếu chẵn, `1` nếu lẻ (với `x >= 0`).
- `x & 1` lấy **bit thấp nhất** của `x` — cũng là `1` khi lẻ, `0` khi chẵn. Nhanh và hay gặp trong code bit.

In [3]:
for x in [0, 1, 2, 7, 10]:
    print(f"x={x:>2}: x % 2 = {x % 2} | x & 1 = {x & 1} | bin = {bin(x)}")

x= 0: x % 2 = 0 | x & 1 = 0 | bin = 0b0
x= 1: x % 2 = 1 | x & 1 = 1 | bin = 0b1
x= 2: x % 2 = 0 | x & 1 = 0 | bin = 0b10
x= 7: x % 2 = 1 | x & 1 = 1 | bin = 0b111
x=10: x % 2 = 0 | x & 1 = 0 | bin = 0b1010


### 2.3. Chia lấy phần nguyên `//`

`7 // 2 = 3`, `8 // 2 = 4`. Với số không âm, `//` làm tròn **xuống**. Đây là công cụ chính của công thức đếm.

In [4]:
for x in [0, 1, 2, 3, 7, 8]:
    print(f"{x} // 2 = {x // 2} | ({x} + 1) // 2 = {(x + 1) // 2}")

0 // 2 = 0 | (0 + 1) // 2 = 0
1 // 2 = 0 | (1 + 1) // 2 = 1
2 // 2 = 1 | (2 + 1) // 2 = 1
3 // 2 = 1 | (3 + 1) // 2 = 2
7 // 2 = 3 | (7 + 1) // 2 = 4
8 // 2 = 4 | (8 + 1) // 2 = 4


## 3. Cách 1 — Brute force: duyệt từng số

In [5]:
def count_odds_bruteforce(low: int, high: int) -> int:
    count = 0
    for x in range(low, high + 1):
        if x % 2 == 1:
            count += 1
    return count

print(count_odds_bruteforce(3, 7))    # 3
print(count_odds_bruteforce(8, 10))   # 1

3
1


## 4. Nút thắt — `10^9` vòng lặp là quá chậm

Brute force chạy `high - low + 1` vòng. Với `low = 0, high = 10^9` là **một tỷ** vòng — Python mất cỡ vài chục giây tới vài phút → **Time Limit Exceeded**. Đo thử với 10 triệu số để ước lượng:

In [6]:
import time
t = time.perf_counter()
count_odds_bruteforce(0, 10**7)
elapsed = time.perf_counter() - t
print(f"10^7 số mất {elapsed:.2f} s -> ước lượng 10^9 số mất khoảng {elapsed * 100:.0f} s")

10^7 số mất 0.36 s -> ước lượng 10^9 số mất khoảng 36 s


Ta cần đếm **mà không duyệt**: một công thức `O(1)`.

## 5. Cách 2 — Bỏ qua số chẵn: `range` bước 2

Cải tiến nhỏ: bắt đầu từ số lẻ đầu tiên `>= low`, nhảy bước 2. `len(range(...))` tính **ngay lập tức** (range không tạo list, Python tính độ dài bằng công thức).

In [7]:
def count_odds_range(low: int, high: int) -> int:
    first_odd = low if low % 2 == 1 else low + 1
    return len(range(first_odd, high + 1, 2))

print(count_odds_range(3, 7), count_odds_range(8, 10))
print("range(1, 10**9 + 1, 2) dài:", len(range(1, 10**9 + 1, 2)), "<- tức thì, không tạo list")

3 1
range(1, 10**9 + 1, 2) dài: 500000000 <- tức thì, không tạo list


Cách này đã `O(1)` nhờ `range` thông minh, nhưng ta vẫn muốn hiểu **công thức thật** đằng sau.

## 6. Insight A — đếm số lẻ trong `[0, x]`

Gọi `f(x)` = số số lẻ trong `[0, x]`. Các số lẻ là `1, 3, 5, ...`, cứ 2 số liên tiếp có đúng 1 số lẻ:

| x | các số lẻ trong [0, x] | f(x) | (x + 1) // 2 |
|---:|---|---:|---:|
| 0 | – | 0 | 0 |
| 1 | 1 | 1 | 1 |
| 2 | 1 | 1 | 1 |
| 3 | 1, 3 | 2 | 2 |
| 4 | 1, 3 | 2 | 2 |
| 7 | 1, 3, 5, 7 | 4 | 4 |

Vậy `f(x) = (x + 1) // 2`.

Khi đó số lẻ trong `[low, high]` = (số lẻ trong `[0, high]`) − (số lẻ trong `[0, low - 1]`):

```
answer = f(high) - f(low - 1) = (high + 1) // 2 - low // 2
```

(vì `f(low - 1) = (low - 1 + 1) // 2 = low // 2`). Đây giống hệt kỹ thuật **prefix sum**: "đếm trong đoạn = đếm tới phải − đếm tới trước trái".

In [8]:
def odds_upto(x: int) -> int:
    # số lượng số lẻ trong [0, x], với x >= -1
    return (x + 1) // 2

for x in range(0, 9):
    assert odds_upto(x) == sum(1 for v in range(0, x + 1) if v % 2)
print("f(x) = (x+1)//2 đúng với x = 0..8")
low, high = 3, 7
print(f"f({high}) - f({low - 1}) = {odds_upto(high)} - {odds_upto(low - 1)} = {odds_upto(high) - odds_upto(low - 1)}")

f(x) = (x+1)//2 đúng với x = 0..8
f(7) - f(2) = 4 - 1 = 3


In [9]:
def count_odds_formula(low: int, high: int) -> int:
    return (high + 1) // 2 - low // 2

print(count_odds_formula(3, 7), count_odds_formula(8, 10), count_odds_formula(0, 10**9))

3 1 500000000


## 7. Insight B — phân tích theo độ dài và chẵn/lẻ hai đầu (gợi ý của đề)

Gọi `length = high - low + 1`.
- Nếu `length` **chẵn**: các số xếp thành `length / 2` cặp liên tiếp `(chẵn, lẻ)` hoặc `(lẻ, chẵn)` → đúng **một nửa** là lẻ: `length // 2`.
- Nếu `length` **lẻ**: còn dư một số. Đoạn khi đó bắt đầu và kết thúc **cùng tính chẵn lẻ** (vì `high - low` chẵn). Nếu `low` lẻ thì số dư là lẻ → `length // 2 + 1`; nếu `low` chẵn → `length // 2`.

Gộp lại: `length // 2 + (1 if length lẻ và low lẻ else 0)`.

Ví dụ:
- `[3, 7]`: length 5 (lẻ), low 3 lẻ → `2 + 1 = 3` ✔.
- `[8, 10]`: length 3 (lẻ), low 8 chẵn → `1 + 0 = 1` ✔.
- `[4, 7]`: length 4 (chẵn) → `2` (5, 7) ✔.

In [10]:
def count_odds_parity(low: int, high: int) -> int:
    length = high - low + 1
    count = length // 2
    if length % 2 == 1 and low % 2 == 1:
        count += 1
    return count

for low, high in [(3, 7), (8, 10), (4, 7), (0, 0), (1, 1)]:
    print(f"[{low}, {high}] -> {count_odds_parity(low, high)} | brute = {count_odds_bruteforce(low, high)}")

[3, 7] -> 3 | brute = 3
[8, 10] -> 1 | brute = 1
[4, 7] -> 2 | brute = 2
[0, 0] -> 0 | brute = 0
[1, 1] -> 1 | brute = 1


Một cách viết khác rất phổ biến: nếu **một trong hai đầu** là lẻ thì cộng thêm 1 vào `(high - low) // 2`:

```python
return (high - low) // 2 + (low % 2 == 1 or high % 2 == 1)
```

`True` / `False` khi cộng vào số được coi là `1` / `0`.

In [11]:
def count_odds_either_end(low: int, high: int) -> int:
    return (high - low) // 2 + (low % 2 == 1 or high % 2 == 1)

print(True + 0, False + 0)
print(count_odds_either_end(3, 7), count_odds_either_end(8, 10), count_odds_either_end(4, 7))

1 0
3 1 2


### 7.1. Phiên bản thao tác bit: `(high - low) // 2 + ((low | high) & 1)`

Điều kiện "một trong hai đầu lẻ" chính là "bit thấp nhất của `low` **hoặc** của `high` bằng 1".
- `low | high` là phép **OR theo bit**: bit kết quả là 1 nếu ít nhất một bên có bit 1.
- `(low | high) & 1` lấy bit thấp nhất của kết quả → `1` nếu ít nhất một đầu lẻ, ngược lại `0`.

Vì sao "một đầu lẻ thì cộng 1" đúng? Trong `high - low` bước nhảy từ `low` tới `high`, cứ 2 bước có đúng 1 lần **đứng trên** số lẻ (không tính điểm xuất phát) — nên `(high - low) // 2` gần đúng; phần chênh lệch chỉ nằm ở hai đầu:

| low | high | `high - low` | `(high-low)//2` | cần cộng | đáp án |
|:-:|:-:|:-:|:-:|:-:|:-:|
| lẻ (3) | lẻ (7) | 4 | 2 | +1 | 3 |
| chẵn (4) | lẻ (7) | 3 | 1 | +1 | 2 |
| lẻ (3) | chẵn (8) | 5 | 2 | +1 | 3 |
| chẵn (8) | chẵn (10) | 2 | 1 | +0 | 1 |

In [12]:
def count_odds_bits(low: int, high: int) -> int:
    return (high - low) // 2 + ((low | high) & 1)

for low, high in [(3, 7), (4, 7), (3, 8), (8, 10)]:
    print(f"[{low}, {high}]: {low} | {high} = {low | high} ({bin(low | high)}), & 1 = {(low | high) & 1}"
          f" -> {count_odds_bits(low, high)} | brute = {count_odds_bruteforce(low, high)}")

[3, 7]: 3 | 7 = 7 (0b111), & 1 = 1 -> 3 | brute = 3
[4, 7]: 4 | 7 = 7 (0b111), & 1 = 1 -> 2 | brute = 2
[3, 8]: 3 | 8 = 11 (0b1011), & 1 = 1 -> 3 | brute = 3
[8, 10]: 8 | 10 = 10 (0b1010), & 1 = 0 -> 1 | brute = 1


## 8. TRACE từng bước — công thức `(high + 1)//2 - low//2`

| low | high | `(high+1)//2` = số lẻ trong [0, high] | `low//2` = số lẻ trong [0, low-1] | Kết quả |
|---:|---:|---:|---:|---:|
| 3 | 7 | 4 (1,3,5,7) | 1 (1) | 3 |
| 8 | 10 | 5 (1,3,5,7,9) | 4 (1,3,5,7) | 1 |
| 0 | 0 | 0 | 0 | 0 |
| 1 | 1 | 1 | 0 | 1 |
| 21 | 22 | 11 | 10 | 1 |

In [13]:
def trace_count_odds(low, high):
    up_to_high = (high + 1) // 2
    before_low = low // 2
    print(f"[{low}, {high}]:")
    print(f"  số lẻ trong [0, {high}]      = ({high} + 1) // 2 = {up_to_high}"
          + (f"  -> {[v for v in range(0, high + 1) if v % 2]}" if high <= 15 else ""))
    print(f"  số lẻ trong [0, {low - 1}]  = {low} // 2 = {before_low}"
          + (f"  -> {[v for v in range(0, low) if v % 2]}" if low <= 15 else ""))
    print(f"  kết quả = {up_to_high} - {before_low} = {up_to_high - before_low}")
    return up_to_high - before_low

trace_count_odds(3, 7)
trace_count_odds(8, 10)
trace_count_odds(21, 22)

[3, 7]:
  số lẻ trong [0, 7]      = (7 + 1) // 2 = 4  -> [1, 3, 5, 7]
  số lẻ trong [0, 2]  = 3 // 2 = 1  -> [1]
  kết quả = 4 - 1 = 3
[8, 10]:
  số lẻ trong [0, 10]      = (10 + 1) // 2 = 5  -> [1, 3, 5, 7, 9]
  số lẻ trong [0, 7]  = 8 // 2 = 4  -> [1, 3, 5, 7]
  kết quả = 5 - 4 = 1
[21, 22]:
  số lẻ trong [0, 22]      = (22 + 1) // 2 = 11
  số lẻ trong [0, 20]  = 21 // 2 = 10
  kết quả = 11 - 10 = 1


1

## 9. Vì sao công thức đúng?

**Bổ đề:** với `x >= -1`, số số lẻ trong `[0, x]` là `(x + 1) // 2`.
- Các số lẻ `<= x` là `1, 3, ..., 2m - 1` với `2m - 1 <= x`, tức `m <= (x + 1) / 2`. Số lớn nhất `m` thỏa mãn là `⌊(x + 1) / 2⌋ = (x + 1) // 2`.
- Với `x = -1` (đoạn rỗng): `0 // 2 = 0` ✔.

**Định lý:** số lẻ trong `[low, high]` = `f(high) - f(low - 1)`, vì `[0, high]` chia thành hai phần rời nhau `[0, low - 1]` và `[low, high]`.

Thay vào: `(high + 1) // 2 - low // 2`. Không có vòng lặp nên không cần invariant — chỉ cần lập luận đếm.

## 10. Code nộp LeetCode

In [14]:
class Solution:
    def countOdds(self, low: int, high: int) -> int:
        return (high + 1) // 2 - low // 2

sol = Solution()
assert sol.countOdds(3, 7) == 3
assert sol.countOdds(8, 10) == 1
print("PASS ví dụ đề")

PASS ví dụ đề


## 11. Phân tích Time / Space

Gọi `n = high - low + 1` (số phần tử của đoạn).

| Cách | Thời gian | Bộ nhớ | Ghi chú |
|---|---|---|---|
| 1. Brute force | `O(n)` | `O(1)` | TLE với `n ≈ 10^9` |
| 2. `len(range(..., 2))` | `O(1)` | `O(1)` | `range` tính độ dài bằng công thức |
| 3. `(high+1)//2 - low//2` | `O(1)` | `O(1)` | **nên dùng** |
| 4. Phân tích chẵn/lẻ | `O(1)` | `O(1)` | theo gợi ý đề |
| 5. Thao tác bit | `O(1)` | `O(1)` | gọn, cần hiểu `|` và `&` |

(Phép `//` trên số nguyên cỡ `10^9` coi là `O(1)`.)

## 12. Kiểm thử: mẫu đề, biên, đối chiếu

In [15]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name:<32} -> {actual}")

tests = [
    ("đề 1 (lẻ - lẻ)",                 3, 7, 3),
    ("đề 2 (chẵn - chẵn)",             8, 10, 1),
    ("chẵn - lẻ",                      4, 7, 2),
    ("lẻ - chẵn",                      3, 8, 3),
    ("một số chẵn",                    0, 0, 0),
    ("một số lẻ",                      1, 1, 1),
    ("0..1",                           0, 1, 1),
    ("toàn miền",                      0, 10**9, 5 * 10**8),
    ("đầu mút lớn, lẻ",                10**9 - 1, 10**9 - 1, 1),
]
fs = (count_odds_range, count_odds_formula, count_odds_parity, count_odds_either_end, count_odds_bits)
for name, low, high, exp in tests:
    for f in fs:
        assert f(low, high) == exp, (f.__name__, low, high)
    check(name, Solution().countOdds(low, high), exp)

PASS  đề 1 (lẻ - lẻ)                   -> 3
PASS  đề 2 (chẵn - chẵn)               -> 1
PASS  chẵn - lẻ                        -> 2
PASS  lẻ - chẵn                        -> 3
PASS  một số chẵn                      -> 0
PASS  một số lẻ                        -> 1
PASS  0..1                             -> 1
PASS  toàn miền                        -> 500000000
PASS  đầu mút lớn, lẻ                  -> 1


In [16]:
# Đối chiếu với brute force trên MỌI cặp 0 <= low <= high <= 200
cnt = 0
for low in range(0, 201):
    for high in range(low, 201):
        exp = count_odds_bruteforce(low, high)
        for f in fs:
            assert f(low, high) == exp, (f.__name__, low, high)
        cnt += 1
print(f"Đã đối chiếu {cnt} cặp (low, high). PASS")

Đã đối chiếu 20301 cặp (low, high). PASS


## 13. Lỗi người mới hay gặp

### Lỗi 1: Quên "+1" của đoạn bao gồm hai đầu
```python
length = high - low          # SAI: [3, 7] có 5 số chứ không phải 4
```

### Lỗi 2: `(high - low) // 2` không xét đầu mút
`[3, 7]` → `4 // 2 = 2`, nhưng đúng là 3 (thiếu vì cả hai đầu đều lẻ).

### Lỗi 3: Công thức `high // 2 - low // 2`
`high // 2 = f(high - 1)`, nên công thức này đếm số lẻ trong `[low, high - 1]` — **bỏ mất `high`**, sai khi `high` lẻ. `[3, 7]` → `3 - 1 = 2`.

### Lỗi 4: `range(low, high)` trong brute force — bỏ mất `high`.

### Lỗi 5: Dùng `/` thay vì `//` → kết quả float như `2.5`, và công thức không còn làm tròn xuống.

In [17]:
low, high = 3, 7
print("đúng                       :", count_odds_formula(low, high))
print("lỗi 1 length thiếu +1      :", (high - low) // 2, "(vô tình dùng length = 4)")
print("lỗi 2 (high-low)//2        :", (high - low) // 2)
print("lỗi 3 high//2 - low//2     :", high // 2 - low // 2)
print("lỗi 4 range(low, high)     :", sum(1 for x in range(low, high) if x % 2))
print("lỗi 5 dùng / với [3, 7]    :", (high + 1) / 2 - low / 2, "<- float và sai giá trị")

đúng                       : 3
lỗi 1 length thiếu +1      : 2 (vô tình dùng length = 4)
lỗi 2 (high-low)//2        : 2
lỗi 3 high//2 - low//2     : 2
lỗi 4 range(low, high)     : 2
lỗi 5 dùng / với [3, 7]    : 2.5 <- float và sai giá trị


Ở lỗi 5: với `[3, 7]` kết quả là `2.5` — vừa sai **kiểu** (float thay vì int) vừa sai **giá trị**, vì `low / 2 = 1.5` không được làm tròn xuống như `low // 2 = 1`. Tương tự `[3, 3]` cho `2 - 1.5 = 0.5` thay vì `1`.

In [18]:
print("[3, 3] dùng / :", (3 + 1) / 2 - 3 / 2, "| dùng // :", (3 + 1) // 2 - 3 // 2)

[3, 3] dùng / : 0.5 | dùng // : 1


## 14. Mẹo nhận dạng pattern

- "Đếm số thỏa tính chất đơn giản trong đoạn `[low, high]` rất lớn" → **không duyệt**; tìm `f(x)` = đếm trong `[0, x]` rồi lấy `f(high) - f(low - 1)`.
- Tính chất tuần hoàn (chẵn/lẻ, chia hết cho `k`, chu kỳ ngày trong tuần…) → `f(x)` thường là `x // k` hoặc `(x + c) // k`.
- Constraints tới `10^9` mà bài Easy → gần như chắc chắn có công thức `O(1)` hoặc `O(log n)`.

Họ bài: 2894 Divisible and Non-divisible Sums Difference, và các bài "đếm số bội của k trong đoạn".

## 15. Biến thể

### Biến thể A — Đếm số chia hết cho `k` trong `[low, high]`
`f(x) = x // k` (số bội dương của `k` trong `[1, x]`; tính cả 0 nếu `low = 0` — xem bài tập 4).

In [19]:
def count_multiples(low, high, k):
    # số bội của k trong [low, high], với low >= 1
    return high // k - (low - 1) // k

print(count_multiples(1, 10, 3), [x for x in range(1, 11) if x % 3 == 0])
print(count_multiples(10, 30, 7), [x for x in range(10, 31) if x % 7 == 0])

3 [3, 6, 9]
3 [14, 21, 28]


### Biến thể B — Tổng các số lẻ trong `[low, high]`
Tổng `m` số lẻ đầu tiên là `1 + 3 + ... + (2m - 1) = m²`. Vậy tổng các số lẻ trong `[0, x]` là `f(x)²`.

In [20]:
def sum_odds(low, high):
    g = lambda x: ((x + 1) // 2) ** 2      # tổng các số lẻ trong [0, x]
    return g(high) - g(low - 1)

print(sum_odds(3, 7), 3 + 5 + 7)
print(sum_odds(8, 10), 9)

15 15
9 9


`lambda x: biểu_thức` tạo **hàm vô danh** ngắn gọn một dòng — tương đương `def g(x): return biểu_thức`.

## 16. Luyện tập — 6 bài từ dễ đến nâng cao

### Bài 1 (tính tay)
Có bao nhiêu số lẻ trong `[14, 39]`? Dùng công thức, trả về kết quả.

In [21]:
def exercise_by_hand() -> int:
    # TODO: (39 + 1) // 2 - 14 // 2 = ?
    pass

# assert exercise_by_hand() == count_odds_bruteforce(14, 39)

### Bài 2 (đếm số chẵn)
Viết `count_evens(low, high)` trong `O(1)`.

In [22]:
def exercise_count_evens(low: int, high: int) -> int:
    # TODO: tổng số phần tử - số lẻ, hoặc tự tìm f(x)
    pass

# assert exercise_count_evens(3, 7) == 2
# assert exercise_count_evens(0, 0) == 1
# assert exercise_count_evens(8, 10) == 2

### Bài 3 (tự suy công thức theo chẵn/lẻ)
Viết lại cách "phân tích chẵn/lẻ hai đầu" mà không nhìn mục 7.

In [23]:
def exercise_count_odds_parity(low: int, high: int) -> int:
    # TODO: length = ?; nếu length lẻ và low lẻ thì ...
    pass

# assert exercise_count_odds_parity(3, 7) == 3
# assert exercise_count_odds_parity(8, 10) == 1
# assert exercise_count_odds_parity(4, 7) == 2

### Bài 4 (bội của `k`, cho phép `low = 0`)
Đếm số `x` trong `[low, high]` với `x % k == 0`, kể cả `x = 0`. `0 <= low <= high`, `k >= 1`.

In [24]:
def exercise_count_multiples(low: int, high: int, k: int) -> int:
    # TODO: f(x) = số bội của k trong [0, x] = ?
    pass

# assert exercise_count_multiples(0, 10, 3) == 4     # 0, 3, 6, 9
# assert exercise_count_multiples(4, 4, 2) == 1
# assert exercise_count_multiples(5, 5, 2) == 0

### Bài 5 (số có dư `r` khi chia `k`)
Đếm `x` trong `[low, high]` với `x % k == r` (`0 <= r < k`).

In [25]:
def exercise_count_remainder(low: int, high: int, k: int, r: int) -> int:
    # TODO: tịnh tiến: x % k == r  <=>  (x - r) % k == 0
    pass

# assert exercise_count_remainder(0, 20, 5, 2) == 4    # 2, 7, 12, 17
# assert exercise_count_remainder(3, 7, 2, 1) == 3     # chính là bài gốc

### Bài 6 (mini challenge — đếm số lẻ không chia hết cho 3)
Đếm số `x` trong `[low, high]` vừa **lẻ** vừa **không chia hết cho 3**, trong `O(1)`.

Gợi ý: số lẻ chia hết cho 3 là số có dư `3` khi chia `6`.

In [26]:
def exercise_odd_not_div3(low: int, high: int) -> int:
    # TODO
    pass

# assert exercise_odd_not_div3(1, 10) == 3          # 1, 5, 7
# assert exercise_odd_not_div3(10, 30) == sum(1 for x in range(10, 31) if x % 2 and x % 3)   # đối chiếu brute force

### Gợi ý ba mức

**Bài 1** — M1: dùng `f(high) - f(low - 1)`. M2: `40 // 2 = 20`, `14 // 2 = 7`. M3: `20 - 7`.

**Bài 2** — M1: tổng số phần tử là `high - low + 1`. M2: số chẵn = tổng − số lẻ. M3: `return (high - low + 1) - countOdds(low, high)`.

**Bài 3** — M1: `length = high - low + 1`. M2: `count = length // 2`. M3: `if length % 2 == 1 and low % 2 == 1: count += 1`.

**Bài 4** — M1: bội của `k` trong `[0, x]` gồm `0, k, 2k, ..., ` → `x // k + 1` số (với `x >= 0`). M2: với `x = -1` thì là `0`. M3: `f(high) - f(low - 1)` với `f(-1) = 0`.

**Bài 5** — M1: đặt `y = x - r`, đếm `y` chia hết cho `k` trong `[low - r, high - r]`. M2: `f(x) = (x - r) // k + 1` nếu `x >= r` ngược lại `0` — đếm trong `[0, x]`. M3: `f(high) - f(low - 1)`.

**Bài 6** — M1: số lẻ − số lẻ chia hết cho 3. M2: số lẻ chia hết cho 3 ⇔ `x % 6 == 3`. M3: dùng bài 5 với `k = 6, r = 3`.

## 17. Đáp án tham khảo (tự làm trước rồi mới xem)

In [27]:
def answer_by_hand():
    return (39 + 1) // 2 - 14 // 2        # 20 - 7 = 13

def answer_count_evens(low, high):
    return (high - low + 1) - count_odds_formula(low, high)

def answer_count_odds_parity(low, high):
    length = high - low + 1
    count = length // 2
    if length % 2 == 1 and low % 2 == 1:
        count += 1
    return count

def answer_count_multiples(low, high, k):
    f = lambda x: x // k + 1 if x >= 0 else 0      # bội của k trong [0, x]
    return f(high) - f(low - 1)

def answer_count_remainder(low, high, k, r):
    f = lambda x: (x - r) // k + 1 if x >= r else 0   # số y trong [0, x] có y % k == r
    return f(high) - f(low - 1)

def answer_odd_not_div3(low, high):
    return count_odds_formula(low, high) - answer_count_remainder(low, high, 6, 3)

assert answer_by_hand() == count_odds_bruteforce(14, 39) == 13
assert answer_count_evens(3, 7) == 2 and answer_count_evens(0, 0) == 1 and answer_count_evens(8, 10) == 2
assert answer_count_odds_parity(3, 7) == 3 and answer_count_odds_parity(8, 10) == 1 and answer_count_odds_parity(4, 7) == 2
assert answer_count_multiples(0, 10, 3) == 4
assert answer_count_multiples(4, 4, 2) == 1 and answer_count_multiples(5, 5, 2) == 0
assert answer_count_remainder(0, 20, 5, 2) == 4
assert answer_count_remainder(3, 7, 2, 1) == 3
assert answer_odd_not_div3(1, 10) == 3
for low in range(0, 60):
    for high in range(low, 60):
        assert answer_count_evens(low, high) == sum(1 for x in range(low, high + 1) if x % 2 == 0)
        assert answer_odd_not_div3(low, high) == sum(1 for x in range(low, high + 1) if x % 2 and x % 3)
        for k in (1, 2, 3, 7):
            assert answer_count_multiples(low, high, k) == sum(1 for x in range(low, high + 1) if x % k == 0)
            for r in range(k):
                assert answer_count_remainder(low, high, k, r) == sum(1 for x in range(low, high + 1) if x % k == r)
print("answer_odd_not_div3(0, 10**9) =", answer_odd_not_div3(0, 10**9))
print("Tất cả đáp án PASS!")

answer_odd_not_div3(0, 10**9) = 333333333
Tất cả đáp án PASS!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def countOdds(self, low: int, high: int) -> int:
        # số lẻ trong [0, x] = (x + 1) // 2  ->  đếm trong đoạn = f(high) - f(low - 1)
        return (high + 1) // 2 - low // 2
```

Khuôn tổng quát: **đếm trong `[low, high]` = `f(high) - f(low - 1)`**, với `f(x)` = đếm trong `[0, x]`.

**Checklist trước khi submit:**
- [ ] Không duyệt vòng lặp khi `high` tới `10^9`?
- [ ] Đoạn **bao gồm hai đầu** (nhớ `+1` / `low - 1`)?
- [ ] Dùng `//`, không dùng `/`?
- [ ] Đã test 4 tổ hợp chẵn/lẻ của hai đầu và `low == high`?
- [ ] Đã test `low = 0`?